# പാഠം 12 - ഏജന്റ് സ്ക്രാച്ച്‌പാഡോടെ ചാറ്റ് ചരിത്രം കുറയ്ക്കൽ

Microsoft Agent Framework ഉപയോഗിച്ച് ദീർഘസംવાદങ്ങളിൽ context എങ്ങനെ കൈകാര്യം ചെയ്യാമെന്ന് ഈ നോട്ട്‌ബുക്ക് കാണിക്കുന്നു. സംഭാഷണങ്ങൾ വളരുമ്പോൾ ടോക്കൺ എണ്ണം വർധിച്ച് മോഡലിന്റെ context വിൻഡോ കടക്കുന്നു. ഇതിന് പരിഹാരമായി **context സംഗ്രഹ പാറ്റേൺ** ഉം **Persistent memory യുള്ള ഏജന്റ് സ്ക്രാച്ച്‌പാഡ്** ഉം ഉപയോഗിക്കുന്നു.

## നിങ്ങൾ പഠിക്കുക:
1. **Context മാനേജ്മെന്റ് എന്തിന് ആവശ്യമാണ്**: ടോക്കൺ പരിധികളും context വിൻഡോകളും മനസ്സിലാക്കൽ
2. **Context-അവയർ ഏജന്റുകൾ**: സ്വന്തം സംഭാഷണ context മാനേജ്ജ് ചെയ്യുന്ന ഏജന്റുകൾ രൂപീകരിക്കൽ
3. **Context സംഗ്രഹ പാറ്റേൺ**: സംഭാഷണ ചരിത്രം ചുരുക്കാനുള്ള ടൂളുകൾ ഉപയോഗിക്കൽ
4. **Agent Scratchpad**: context കുറയ്ക്കലിനുള്ള persistent memory

## മുൻകൂർ requisitos:
- പരിസ്ഥിതി ചേരുവകൾ കൊടുത്ത് Azure OpenAI സെറ്റപ്പ്
- മുന്‍പത്തെ പാഠങ്ങളിൽ നിന്നുള്ള അടിസ്ഥാന ഏജന്റ് ആശയങ്ങളുടെ അവബോധം


## ക്രമീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## കോൺടെക്‌സ്‌റ്റ് മാനേജ്മെന്റ് എന്തുകൊണ്ട് പ്രധാനമാണ്

ഓരോ LLM നും ഒരു പരിമിതമായ **കോൺടെക്‌സ്‌റ്റ് വിൻഡോ** ഉണ്ട് — ഒരു അഭ്യർത്ഥനയിൽ പ്രോസസ്സ് ചെയ്യാവുന്ന ടോക്കൻസുകളുടെ പരമാവധി എണ്ണം. പലതവണ സംവാദം പുരോഗമിച്ചപ്പോൾ:

- **ടോക്കൻ കൗണ്ട് പോരെഴുതിയ പോലെ വർധിക്കുന്നു** ഓരോ ഉപയോക്തൃ സന്ദേശത്തിലും അസിസ്റ്റന്റ് മറുപടിയിലും.
- **പ്രോംപ്റ്റ് ടോക്കൻസുകൾ ചെലവിൽ ആഴ്ചയ്ക്കും** കാരണം മുഴുവൻ ചരിത്രം ഓരോ തവണയും വീണ്ടും അയയ്ക്കപ്പെടുന്നു.
- അവസാനം സംവാദം **കോൺടെക്‌സ്‌റ്റ് വിൻഡോ മറികടന്ന്** മോഡൽ കേട്ട് കുറയ്ക്കുകയോ പിശകു കാണിക്കുകയോ ചെയ്യും.

### കോൺടെക്‌സ്‌റ്റ് മാനേജ്മെൻറ് രണനീതികൾ

| രണനി‌തി | എങ്ങനെ പ്രവർത്തിക്കുന്നു | നൽകൽ-പടം |
|---|---|---|
| **കുറയ്ക്കൽ** | പഴയ സന്ദേശങ്ങൾ drop ചെയ്യുക | പ്രാരംഭ കോൺടെക്‌സ്‌റ്റ് നഷ്ടം |
| **സംക്ഷേപം** | പഴയ സന്ദേശങ്ങൾ സംക്ഷിപ്തമായി മാറ്റുക | ചില വിശദാംശം നഷ്ടമാകാം, പക്ഷേ പ്രധാന പോയിന്റുകൾ നിലനിർത്തുന്നു |
| **Scratchpad / പുറം മെമ്മറി** | സംഭാഷണത്തിന് പുറത്തുള്ള പ്രധാന വസ്തുതകൾ സൂക്ഷിക്കുക | ടൂൾ വിളിപ്പിക്കലുകൾ ആവശ്യമാണ്, പക്ഷേ ഏതൊരു കുറച്ചിലും നിലനിർത്തും |

ഈ നോട്ട്‌ബുക്കിൽ നാം **സംക്ഷേപം** **scratchpad ടൂൾ**-ഉം ചേർത്ത് ഉപയോക്താവിന് സംവാദതത്വം നിലനിർത്താൻ സഹായിക്കുന്നു, ചുരുങ്ങിയ ചരിത്രത്തിലും.


## കോൺടെക്സ്റ്റ്-അവിയർ ഏജന്റ് സൃഷ്ടിക്കൽ


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## ഒരു ദീർഘസംവാദം അനുകരിക്കുന്നത്

കൂട്ടുകൂട്ടിയ സംഭാഷണത്തിൽ നിലനിൽക്കുന്ന പശ്ചാത്തലം എങ്ങനെ കൂട്ടിയിടുന്നുവെന്ന് കാണാൻ നമുക്ക് ഒന്നു നോക്കാം. ഏജൻ്റ് ചുറ്റുമുള്ള തിരുവനന്തപുരത്തിലെ പ്രധാനമായ വിവരങ്ങൾ (പ്രിയപ്പെട്ടവ, ബജറ്റ്, യാത്രാ തീയതികൾ) നിലനിർത്തുകയും തുടർച്ച കാണിക്കുകയും വേണം.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ഏജന്റ് ആദ്യത്തെ തിരുപ്പുകളിൽ നിന്ന്_context_ നിലനിർത്തുന്ന വിധം ശ്രദ്ധിക്കുക — ജപ്പാൻ, സുഷി, ക്ഷേത്രങ്ങൾ, ഫോട്ടോഗ്രാഫി, $3000 ബജറ്റ്, സോളോ യാത്ര, ഏപ്രിൽ സമയപരിധി എന്നിവയെ കുറിച്ച് അവക്ക് അറിയാം. ഒരു ചരടായ സംഭാഷണത്തിൽ ഇത് നന്നായി പ്രവർത്തിക്കുന്നു, പക്ഷേ സംഭാഷണം വളരുമ്പോൾ മുഴുവൻ ചരിത്രം വീണ്ടും അയയ്ക്കുന്നത് ചെലവേറിയതാകുന്നു.

കൂടുതൽ തിരുപ്പുകളോടെ_Context_ ഏകീകരണം കാണാൻ സംഭാഷണം തുടരാം:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## വഴിക്കുറിപ്പിന്റെ ചുരുക്കം മാതൃക

സംഭാഷണം വളരുന്നതിനോടൊപ്പം, ഞങ്ങൾ **ചുരുക്കൽ ഉപകരണം** ഉപയോഗിച്ച് സമാഹരിച്ച ഉള്ളടക്കം ചുരുക്കിയ ഫോർമാറ്റിലേക്ക് മുറുക്കാം. പുരാതന സന്ദേശങ്ങൾ ഒഴിവാക്കപ്പെട്ടാലും പ്രധാന വിവരങ്ങൾ സംരക്ഷിക്കപ്പെടാനായി ഏജന്റ് ഈ ഉപകരണത്തെ പ്രധാന ഇഷ്ടാനുസൃതികൾ രേഖപ്പെടുത്താൻ വിളിക്കുന്നു.

ഈ മാതൃക കൂടുതൽ സങ്കീർണമായ ചരിത്ര കുറവിന്റെ അടിസ്ഥാനം ആണ്:
1. ഏജന്റ് സംഭാഷണത്തിൽ നിന്നുള്ള പ്രധാന വിവരങ്ങൾ തിരിച്ചറിയുന്നു
2. അവയെ സൂക്ഷിക്കാൻ ചുരുക്കൽ ഉപകരണം വിളിക്കുന്നു
3. ചോദ്യ വാചകങ്ങൾ സുരക്ഷിതമായി നീക്കം ചെയ്യാം കാരണം ചുരുക്കം ആവശ്യമായ കാര്യങ്ങൾ പകർന്നു

താഴെ, ഏജന്റ് പഠിച്ച കാര്യങ്ങളുടെ ചുരുക്ക സംഗ്രഹം രേഖപ്പെടുത്താൻ കഴിവുള്ള `summarize_preferences` ഉപകരണം നാം നിർവചിക്കുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## സംക്ഷേപം

ഈ അധ്യായത്തിൽ, നിങ്ങൾ Microsoft Agent Framework ഉപയോഗിച്ച് ദീർഘകാല പ്രവർത്തന.agent സംഭാഷണങ്ങളിൽ കോൺടെക്സ്റ്റ് എങ്ങനെ നിയന്ത്രിക്കാമെന്ന് പഠിച്ചു:

### പ്രധാന ആശയങ്ങൾ
- **കോൺടെക്സ്റ്റ് വിൻഡോകൾ സീമിതമാണ്** — സംഭാഷണ ചരിത്രത്തിലെ ഓരോ ടോക്കൺ‍ക്കും ചിലവാണ് ഉണ്ടാകുന്നത്, അത് പരിധിയിലേക്ക് հաշվാം.
- **സംക്ഷേപ ഉപകരണങ്ങൾ** ഏജൻറ് സമാഹരിച്ച കോൺടെക്സ്റ്റ് സങ്കുചിത സംഖ്യാപരിപാടികളായി മാറ്റാൻ അനുവദിക്കുന്നു, ടോക്കൺ ഉപയോഗം കുറയ്ക്കുകയും അടിസ്ഥാന വിവരങ്ങൾ നിലനിർത്തുകയും ചെയ്യുന്നു.
- **ഏജന്റ് സ്ക്രാച്ച്പാഡുകൾ** സ്ഥിരമായ ബാഹ്യ സ്മരണം നൽകുന്നു, അത് സംഭാഷണ കുറവുള്ളതും കടന്നുപോകുന്നു.

### നിങ്ങൾ നിർമ്മിച്ചത്
- **കോൺടെക്സ്റ്റ്-അവയർ ഏജന്റ്** മൾട്ടി-ടേൺ സംഭാഷണങ്ങളിൽ തുടർച്ച നിലനിർത്തുന്നു
- പ്രധാന ഉപഭോക്തൃ വിശദാംശങ്ങൾ സംക്ഷിപ്ത രൂപത്തിലാണ് രേഖപ്പെടുത്തുന്ന **സംക്ഷേപ ഉപകരണം** (`summarize_preferences`)
- കോൺടെക്സ്റ്റ് നിലനിർത്തലും മാറ്റം കൈകാര്യം ചെയ്യലും കാണിക്കുന്ന **മൾടി-ടേർൺ സംഭാഷണം**

### യാഥാർത്ഥ്യ ഉപയോഗികൾ
- **ഹരീകാർ സേവന ബോട്ടുകൾ**: ദീർഘക്കാല സഹായ സെഷനുകളിൽ പ്രിഫറൻസുകൾ ഓർമിക്കുക
- **വ്യക്തിഗത അസിസ്റ്റന്റുകൾ**: തുടർ പ്രോജക്റ്റുകൾ കൊണ്ട് കോൺടെക്സ്റ്റ് വീണ്ടും വിശദീകരിക്കാതെ ട്രാക്ക് ചെയ്യുക
- **വിദ്യാഭ്യാസ ട്യൂട്ടർമാർ**: പല ഇന്ററാക്ഷനുകളിലൂടെയുള്ള വിദ്യാർത്ഥി പുരോഗതി നിലനിർത്തുക

### അടുത്ത പടികൾ
- ഫയൽ അടിസ്ഥാനമുള്ള സ്ഥിരതയോടെ ഒരു പൂർണ്ണ സ്ക്രാച്ച്പാഡ് ഉപകരണം നടപ്പിലാക്കുക
- സംക്ഷേപത്തിലേക്കു ശേഷം ഓട്ടോമാറ്റിക് ചരിത്രം കുറയ്ക്കൽ ചേർക്കുക
- സിമാന്റിക് മെമ്മറി സർച്ചിനായി വെക്‌ටർ ഡേറ്റാബേസുകളുമായി പങ്കിടുക
- പൂർണ്ണ കോൺടെക്സ്റ്റോടെ ദിവസങ്ങൾ കഴിഞ്ഞ് സംഭാഷണങ്ങൾ വീണ്ടും തുടങ്ങാൻ കഴിയുന്ന ഏജൻറുകൾ നിർമ്മിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
